## accuracy check

In [3]:
# %% [markdown]
# Per-stratum accuracy + per-factor PR/F1 + overall (micro/macro) PR/F1
# Input:  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10_output.xlsx
# Output: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\validation_metrics.xlsx

# %%
import pandas as pd
import numpy as np
from pathlib import Path

# ---------- CONFIG ----------
FOLDER = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final")
IN_XLSX = FOLDER / "stratified_sample_MR_Final.xlsx"
OUT_XLSX = FOLDER / "validation_metrics.xlsx"

# ---------- Helpers ----------
def to01(s: pd.Series) -> pd.Series:
    """Robust boolean -> {0,1}."""
    if s.dtype == bool:
        return s.astype(int)
    if pd.api.types.is_numeric_dtype(s):
        return (pd.to_numeric(s, errors="coerce").fillna(0) != 0).astype(int)
    t = s.astype(str).str.strip().str.lower()
    truthy = {"1","true","t","yes","y","on"}
    falsy  = {"0","false","f","no","n","off","","none","null","nan"}
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[t.isin(truthy)] = 1
    out[t.isin(falsy)]  = 0
    # fallback numeric
    num = pd.to_numeric(t.str.replace(r"[^0-9\.\-]+","", regex=True), errors="coerce")
    out = out.where(out.notna(), (num.fillna(0) != 0).astype(int))
    return out.astype(int)

def pr_from_counts(tp, fp, fn):
    prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    rec  = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1   = (2*prec*rec)/(prec+rec) if (pd.notna(prec) and pd.notna(rec) and (prec+rec)>0) else np.nan
    return prec, rec, f1

def counts_for(pred, true):
    tp = int(((pred==1) & (true==1)).sum())
    fp = int(((pred==1) & (true==0)).sum())
    fn = int(((pred==0) & (true==1)).sum())
    tn = int(((pred==0) & (true==0)).sum())
    return tp, fp, fn, tn

# ---------- Load ----------
df = pd.read_excel(IN_XLSX)

need = ["YAML_pred","Build_pred","AT_pred","YAML_Check","Build_Check","AT_Check"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"Missing column(s) in input: {missing}")

# Normalize to 0/1
Yp = to01(df["YAML_pred"]);  Bp = to01(df["Build_pred"]);  Ap = to01(df["AT_pred"])
Yt = to01(df["YAML_Check"]); Bt = to01(df["Build_Check"]); At = to01(df["AT_Check"])

# Ensure stratum label (from predictions if not provided)
if "stratum" in df.columns:
    strata = df["stratum"].astype(str)
else:
    strata = pd.Series([f"Y{y}_B{b}_A{a}" for y,b,a in zip(Yp,Bp,Ap)], index=df.index)

# ---------- 1) Per-stratum accuracy (exact triplet match) ----------
triplet_match = (Yp.eq(Yt) & Bp.eq(Bt) & Ap.eq(At)).astype(int)
per_stratum = (
    pd.DataFrame({"stratum": strata, "match": triplet_match})
      .groupby("stratum", dropna=False)
      .agg(n=("match","size"), matches=("match","sum"))
      .reset_index()
      .sort_values("stratum", ignore_index=True)
)
per_stratum["accuracy"] = per_stratum["matches"] / per_stratum["n"]
overall_triplet_accuracy = float(triplet_match.mean())

# ---------- 2) Per-factor precision/recall/F1 (whole sample) ----------
rows = []
for label, pred, true in [("YAML",Yp,Yt), ("Build",Bp,Bt), ("AT",Ap,At)]:
    tp, fp, fn, tn = counts_for(pred, true)
    prec, rec, f1 = pr_from_counts(tp, fp, fn)
    rows.append({
        "factor": label,
        "N": int(len(pred)),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(prec,4) if pd.notna(prec) else np.nan,
        "recall":    round(rec, 4) if pd.notna(rec) else np.nan,
        "f1":        round(f1,   4) if pd.notna(f1) else np.nan,
    })
per_factor_metrics = pd.DataFrame(rows)

# ---------- 3) OVERALL precision/recall/F1 (whole sample) ----------
# Micro-average (pool all Y,B,AT decisions):
tp = per_factor_metrics["TP"].sum()
fp = per_factor_metrics["FP"].sum()
fn = per_factor_metrics["FN"].sum()
prec_micro, rec_micro, f1_micro = pr_from_counts(tp, fp, fn)

# Macro-average (unweighted mean across factors):
prec_macro = per_factor_metrics["precision"].mean(skipna=True)
rec_macro  = per_factor_metrics["recall"].mean(skipna=True)
f1_macro   = per_factor_metrics["f1"].mean(skipna=True)

overall_metrics = pd.DataFrame([
    {"overall_type":"micro", "precision":round(prec_micro,4), "recall":round(rec_micro,4), "f1":round(f1_micro,4)},
    {"overall_type":"macro", "precision":round(prec_macro,4),  "recall":round(rec_macro,4),  "f1":round(f1_macro,4)},
    {"overall_type":"triplet_exact_accuracy", "precision":np.nan, "recall":np.nan, "f1":np.nan}
])
# Attach the triplet exact-match accuracy as a side note:
overall_note = pd.DataFrame([{"overall_triplet_exact_accuracy": round(overall_triplet_accuracy,4),
                              "rows": len(df)}])

# ---------- Save ----------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    per_stratum.to_excel(xw, sheet_name="Stratum_Accuracy", index=False)
    per_factor_metrics.to_excel(xw, sheet_name="Per_Factor_PRF1", index=False)
    overall_metrics.to_excel(xw, sheet_name="Overall_PRF1", index=False)
    overall_note.to_excel(xw, sheet_name="Overall_Note", index=False)

print(f"[OK] Saved metrics → {OUT_XLSX}")
print("Overall (micro) P/R/F1:", round(prec_micro,4), round(rec_micro,4), round(f1_micro,4))
print("Overall (macro) P/R/F1:", round(prec_macro,4), round(rec_macro,4), round(f1_macro,4))
print("Triplet exact-match accuracy:", round(overall_triplet_accuracy,4))


[OK] Saved metrics → C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\validation_metrics.xlsx
Overall (micro) P/R/F1: 1.0 0.9873 0.9936
Overall (macro) P/R/F1: 1.0 0.9849 0.9923
Triplet exact-match accuracy: 0.9816
